# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneebulhaq02/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muneebulhaq02/flyrank-ml-internship"
REPO_DIR = "FlyRank-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())

Working dir: /content/FlyRank-Internship


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The paper reports that Average Position was the most important feature for predicting the Health Score using a Random Forest model. However, the paper also explains that Health Score is partly constructed from Average Position and Impressions.

## Methodology Question

Since Average Position is already part of the Health Score calculation, how much of the reported feature importance reflects genuine predictive ability versus simply reproducing the formula used to build the target? A comparison using an independent target could strengthen this conclusion.

## Finding 2

The paper reports a Logistic Regression model with 71% holdout accuracy for distinguishing growing and declining pages.

## Methodology Question

The paper reports holdout accuracy but provides limited discussion of grouped or time-aware validation. Because content from the same client may appear in both training and testing data, evaluating with grouped or time-based validation could provide a more realistic estimate of model performance in deployment.





In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_findings = {
    "Finding 1":
        "Average Position is reported as the strongest feature for predicting Health Score.",
    "Methodology Question 1":
        "Is this importance partly caused because Average Position is already used inside the Health Score formula?",

    "Finding 2":
        "Logistic Regression achieved approximately 71% holdout accuracy.",
    "Methodology Question 2":
        "Would grouped or time-aware validation produce similar performance?"
}

for k,v in paper_findings.items():
    print(f"{k}:")
    print(v)
    print()

Finding 1:
Average Position is reported as the strongest feature for predicting Health Score.

Methodology Question 1:
Is this importance partly caused because Average Position is already used inside the Health Score formula?

Finding 2:
Logistic Regression achieved approximately 71% holdout accuracy.

Methodology Question 2:
Would grouped or time-aware validation produce similar performance?



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 Random Forest model was first evaluated using a standard random train/test split.

To better simulate deployment, the model was also evaluated using GroupKFold, where pages belonging to the same client remain together. This reduces information leakage caused by similar pages appearing in both training and testing.

Both evaluations are reported below to show the effect of using a more realistic validation strategy.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

X = df[
    [
        "impressions_90d",
        "content_age_days",
        "avg_position",
        "ctr"
    ]
]

y = df["is_declining_label"]

groups = df["client_id"]

# -------------------------
# Random split
# -------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

rf = RandomForestClassifier(
    random_state=42
)

rf.fit(X_train, y_train)

random_accuracy = accuracy_score(
    y_test,
    rf.predict(X_test)
)

# -------------------------
# Honest grouped split
# -------------------------

gkf = GroupKFold(n_splits=5)

train_idx, test_idx = next(
    gkf.split(X, y, groups)
)

rf.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

group_accuracy = accuracy_score(
    y.iloc[test_idx],
    rf.predict(X.iloc[test_idx])
)

print("Random Split Accuracy :", round(random_accuracy,3))
print("Grouped Split Accuracy:", round(group_accuracy,3))

Random Split Accuracy : 0.6
Grouped Split Accuracy: 0.889


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature set was reviewed for possible leakage.

The following checks were performed:

* label-derived variables were excluded
* future information was excluded
* product flags were excluded
* client identifiers were used only for grouped validation and never as model features

This follows the leakage guidance used throughout the internship.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = [
    "trend_pct",
    "trend_direction",
    "health_score",
    "client_id",
    "content_id"
]

print("Leakage Audit")
print("----------------")

for c in excluded:
    print("Excluded:", c)

print()

print("Model Features")

for c in X.columns:
    print("-", c)

print()

print("No future-window variables detected.")
print("No product flags used.")
print("No label-derived columns used.")

Leakage Audit
----------------
Excluded: trend_pct
Excluded: trend_direction
Excluded: health_score
Excluded: client_id
Excluded: content_id

Model Features
- impressions_90d
- content_age_days
- avg_position
- ctr

No future-window variables detected.
No product flags used.
No label-derived columns used.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claims = {
    "Original":
        "The model predicts which pages should be refreshed.",

    "Rewritten":
        "The model identifies pages associated with decline and provides a decision-support ranking for manual review."
}

for k,v in claims.items():
    print(k)
    print(v)
    print()

Original
The model predicts which pages should be refreshed.

Rewritten
The model identifies pages associated with decline and provides a decision-support ranking for manual review.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.